# Uniform Space-Filling in 5D

This notebook demonstrates a two-stage uniform space-filling workflow in a 5D input space. The problem demonstrated is referenced to

https://foqus.readthedocs.io/en/stable/chapt_sdoe/examples-uniform.html#example-usf-3-a-uniform-space-filling-design-for-a-carbon-capture-example-in-a-5-d-input-space

The main idea is the same as in lower dimensions:

- a **candidate set** defines the feasible input combinations,
- a **design** selects a subset of those combinations,
- a **sequential** design adds new runs to improve the combined set.

In this example the candidate set is irregular rather than a simple box, so the table of validated candidate points is an important part of the problem definition.

The workflow has two stages:

- **Stage 1:** build minimax designs of size 10, 11, and 12.
- **Stage 2:** treat the chosen 12-run design as previous data and add 6 more minimax runs.

The notebook uses the the following API calls: `load_csv`, `prepare_design_setup`, `estimate_uniform_runtime`, `design_uniform_batch`, and `plot_pair_matrix`. Files created during the run are saved under `examples/temp/nb_output`.

In [ ]:
from __future__ import annotations

import json
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    elif (ROOT.parent.parent / "src").exists():
        ROOT = ROOT.parent.parent
    else:
        raise RuntimeError("Could not locate the project root.")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from idaes_sdoe import ColumnRoles, load_csv, prepare_design_setup, write_csv
from idaes_sdoe.design import design_uniform_batch, estimate_uniform_runtime
from idaes_sdoe.plotting import plot_pair_matrix, suggest_histogram_bin_map, write_figure

DATA_FILE = ROOT / "examples" / "supporting_data" / "Candidate Points 8perc.csv"
INPUT_COLUMNS = ["G", "lldg", "CapturePerc", "L", "SteamFlow"]
STAGE1_SIZES = [10, 11, 12]
STAGE1_SELECTED_SIZE = 12
STAGE2_SIZE = 6
NUM_RESTARTS = 10_000
CALIBRATION_RESTARTS = 100
HISTOGRAM_TARGET_BINS = 25

def make_output_dir(label: str) -> Path:
    stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    path = ROOT / "examples" / "temp" / "nb_output" / f"{stamp}_{label}"
    path.mkdir(parents=True, exist_ok=True)
    return path

def write_json(payload: dict[str, object], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)

def save_figure(figure, path: Path) -> None:
    try:
        write_figure(figure, path)
    except Exception as exc:
        print(f"warning: could not save {path.name}: {exc}")

output_dir = make_output_dir("example_uniform_5d_nb")
output_dir

## 1. Load and inspect the 5D candidate set

The candidate set defines the feasible operating region in five variables. Before searching for a design, it is useful to inspect the table, confirm the active input columns, and visualize the pairwise structure of the candidate region.

In [ ]:
candidate = load_csv(DATA_FILE).rename(columns={"Steam Flow": "SteamFlow"})
stage1_setup = prepare_design_setup(
    candidate=candidate,
    roles=ColumnRoles(inputs=INPUT_COLUMNS),
    auto_index=True,
    index_column="__id",
)

write_csv(stage1_setup.candidate, output_dir / "candidate_set.csv")
display(Markdown(f"**Rows:** {len(stage1_setup.candidate)}  \n**Inputs:** {', '.join(INPUT_COLUMNS)}"))
display(stage1_setup.candidate.head())
display(stage1_setup.candidate[INPUT_COLUMNS].agg(["min", "max"]))

candidate_bins = suggest_histogram_bin_map(
    stage1_setup.candidate[INPUT_COLUMNS],
    target_bins=HISTOGRAM_TARGET_BINS,
)
candidate_fig = plot_pair_matrix(
    stage1_setup.candidate[INPUT_COLUMNS],
    columns=INPUT_COLUMNS,
    title="USF-3 candidate set pairwise plot",
    histogram_bins=candidate_bins,
)
save_figure(candidate_fig, output_dir / "candidate_pairwise.pdf")
candidate_fig

## 2. Run the first stage

The first stage creates several minimax alternatives from the same candidate set. This is a common design step: generate a small set of candidate designs, compare cost versus coverage, and then choose one design to execute.

In [ ]:
stage1_dir = output_dir / "stage1_minimax"
stage1_dir.mkdir(parents=True, exist_ok=True)

stage1_estimate = estimate_uniform_runtime(
    setup=stage1_setup,
    design_sizes=STAGE1_SIZES,
    target_restarts=NUM_RESTARTS,
    mode="minimax",
    calibration_restarts=CALIBRATION_RESTARTS,
)
write_json(stage1_estimate, stage1_dir / "runtime_estimate.json")

stage1_started = time.perf_counter()
stage1_results = design_uniform_batch(
    setup=stage1_setup,
    design_sizes=STAGE1_SIZES,
    num_restarts=NUM_RESTARTS,
    mode="minimax",
)
stage1_elapsed = time.perf_counter() - stage1_started

stage1_rows = []
stage1_results_by_size = {}
design_dir = stage1_dir / "designs"
plot_dir = stage1_dir / "plots"
design_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)

for result in stage1_results:
    stage1_results_by_size[result.design_size] = result
    design_file = design_dir / f"stage1_minimax_design_{result.design_size:02d}.csv"
    pair_file = plot_dir / f"stage1_minimax_design_{result.design_size:02d}_pairwise.pdf"
    write_csv(result.design, design_file)

    bins = suggest_histogram_bin_map(
        result.design[INPUT_COLUMNS],
        reference=stage1_setup.candidate[INPUT_COLUMNS],
        target_bins=HISTOGRAM_TARGET_BINS,
    )
    fig = plot_pair_matrix(
        result.design[INPUT_COLUMNS],
        columns=INPUT_COLUMNS,
        candidate=stage1_setup.candidate[INPUT_COLUMNS],
        title=(
            f"Stage 1 minimax design, size {result.design_size}, "
            f"criterion={result.criterion_value:.6f}"
        ),
        histogram_bins=bins,
    )
    save_figure(fig, pair_file)

    stage1_rows.append(
        {
            "design_size": result.design_size,
            "criterion_value": result.criterion_value,
            "elapsed_time": result.elapsed_time,
            "design_file": str(design_file),
            "pair_plot_pdf": str(pair_file),
        }
    )

write_json(
    {
        "mode": "minimax",
        "design_sizes": STAGE1_SIZES,
        "num_restarts": NUM_RESTARTS,
        "calibration_restarts": CALIBRATION_RESTARTS,
        "elapsed_seconds": stage1_elapsed,
    },
    stage1_dir / "run_summary.json",
)

stage1_summary = pd.DataFrame(stage1_rows)
write_csv(stage1_summary, output_dir / "stage1_created_designs_summary.csv")
stage1_summary

In [ ]:
display(Markdown("**Stage 1 design plots**"))
for size in STAGE1_SIZES:
    result = stage1_results_by_size[size]
    bins = suggest_histogram_bin_map(
        result.design[INPUT_COLUMNS],
        reference=stage1_setup.candidate[INPUT_COLUMNS],
        target_bins=HISTOGRAM_TARGET_BINS,
    )
    fig = plot_pair_matrix(
        result.design[INPUT_COLUMNS],
        columns=INPUT_COLUMNS,
        candidate=stage1_setup.candidate[INPUT_COLUMNS],
        title=(
            f"Stage 1 minimax design, size {size}, "
            f"criterion={result.criterion_value:.6f}"
        ),
        histogram_bins=bins,
    )
    display(fig)


## 3. Choose the 12-run first stage and set up the augmentation

After the first stage is assumed complete, the selected 12-run design becomes **previous data**. The second-stage setup keeps the same five design inputs and evaluates new points relative to both the remaining candidate set and the runs that already exist.

In [ ]:
previous_design = stage1_results_by_size[STAGE1_SELECTED_SIZE].design.copy()
write_csv(previous_design, output_dir / "stage1_selected_design.csv")

stage2_setup = prepare_design_setup(
    candidate=stage1_setup.candidate,
    previous=previous_design,
    roles=ColumnRoles(index="__id", inputs=INPUT_COLUMNS),
    auto_index=False,
)

write_csv(stage2_setup.previous, output_dir / "stage2_previous_data.csv")
setup_bins = suggest_histogram_bin_map(
    stage2_setup.previous[INPUT_COLUMNS],
    reference=stage2_setup.candidate[INPUT_COLUMNS],
    target_bins=HISTOGRAM_TARGET_BINS,
)
setup_fig = plot_pair_matrix(
    stage2_setup.previous[INPUT_COLUMNS],
    columns=INPUT_COLUMNS,
    candidate=stage2_setup.candidate[INPUT_COLUMNS],
    title="Stage 2 setup: previous data against candidate set",
    histogram_bins=setup_bins,
)
save_figure(setup_fig, output_dir / "stage2_setup_pairwise.pdf")
setup_fig

## 4. Run the second stage

The second stage adds 6 minimax runs. The design criterion is evaluated on the combined set of old and new points, so the new runs are chosen for how well they improve the full 18-run design rather than how they perform in isolation.

In [ ]:
stage2_dir = output_dir / "stage2_minimax"
stage2_dir.mkdir(parents=True, exist_ok=True)

stage2_estimate = estimate_uniform_runtime(
    setup=stage2_setup,
    design_sizes=[STAGE2_SIZE],
    target_restarts=NUM_RESTARTS,
    mode="minimax",
    calibration_restarts=CALIBRATION_RESTARTS,
)
write_json(stage2_estimate, stage2_dir / "runtime_estimate.json")

stage2_started = time.perf_counter()
stage2_result = design_uniform_batch(
    setup=stage2_setup,
    design_sizes=[STAGE2_SIZE],
    num_restarts=NUM_RESTARTS,
    mode="minimax",
)[0]
stage2_elapsed = time.perf_counter() - stage2_started

total_size = len(stage2_setup.previous) + stage2_result.design_size
design_dir = stage2_dir / "designs"
plot_dir = stage2_dir / "plots"
design_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)

additional_file = design_dir / f"stage2_minimax_additional_{stage2_result.design_size:02d}.csv"
combined_file = design_dir / f"stage2_minimax_combined_total_{total_size:02d}.csv"
pair_file = plot_dir / f"stage2_minimax_additional_{stage2_result.design_size:02d}_pairwise.pdf"

combined_design = pd.concat((stage2_setup.previous, stage2_result.design), ignore_index=True)
write_csv(stage2_result.design, additional_file)
write_csv(combined_design, combined_file)

stage2_bins = suggest_histogram_bin_map(
    stage2_result.design[INPUT_COLUMNS],
    reference=stage2_setup.candidate[INPUT_COLUMNS],
    target_bins=HISTOGRAM_TARGET_BINS,
)
stage2_fig = plot_pair_matrix(
    stage2_result.design[INPUT_COLUMNS],
    columns=INPUT_COLUMNS,
    candidate=stage2_setup.candidate[INPUT_COLUMNS],
    previous=stage2_setup.previous[INPUT_COLUMNS],
    title=(
        f"Stage 2 minimax augmentation, +{stage2_result.design_size} "
        f"(total {total_size}), criterion={stage2_result.criterion_value:.6f}"
    ),
    histogram_bins=stage2_bins,
)
save_figure(stage2_fig, pair_file)

stage2_summary = pd.DataFrame(
    [
        {
            "additional_design_size": stage2_result.design_size,
            "total_design_size": total_size,
            "criterion_value": stage2_result.criterion_value,
            "elapsed_time": stage2_result.elapsed_time,
            "additional_design_file": str(additional_file),
            "combined_design_file": str(combined_file),
            "pair_plot_pdf": str(pair_file),
        }
    ]
)
write_csv(stage2_summary, output_dir / "stage2_created_designs_summary.csv")
write_json(
    {
        "mode": "minimax",
        "additional_design_size": STAGE2_SIZE,
        "num_restarts": NUM_RESTARTS,
        "calibration_restarts": CALIBRATION_RESTARTS,
        "elapsed_seconds": stage2_elapsed,
    },
    stage2_dir / "run_summary.json",
)
display(stage2_summary)
stage2_fig